In [38]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType, TimestampType
from pyspark.sql.functions import col, trim, to_date, to_timestamp

spark = (
    SparkSession.builder
    .appName("Test")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)
spark

In [94]:
schema = StructType([
  StructField("order_id", StringType(), False),
  StructField("cust_id", StringType(), False),
  StructField("product_name", StringType(), True),
  StructField("price", DoubleType(), True),
  StructField("status", StringType(), True),
  StructField("order_date", StringType(), True),
  StructField("updated_at", StringType(), True)
])

cols = ["order_id", "cust_id", "product_name", "price", "status", "order_date", "updated_at"]

target_data = [
    ("ORD-001", "C100", "Laptop",      1200.00, "shipped",    "2025-01-15", "2025-01-16"),
    ("ORD-002", "C101", "Mouse ",         25.50, "delivered",  "2025-01-20", "2025-01-22"),
    ("ORD-003", "C102", " Keyboard",      75.00, "processing", "2025-02-01", "2025-02-01"),
    ("ORD-004", "C100", "Monitor",      450.00, "shipped",    "2025-02-10", "2025-02-11"),
    ("ORD-005", "C103", "Headphones",   150.00, "delivered",  "2025-02-14", "2025-02-16"),
    ("ORD-006", "C104", "Webcam",        89.99, "processing", "2025-03-01", "2025-03-01"),
    ("ORD-007", "C101", "USB Hub",       35.00, "shipped",    "2025-03-05", "2025-03-06"),
    ("ORD-008", "C105", "Tablet",       599.00, "processing", "2025-03-10", "2025-03-10"),
]

incremental_data = [
    ("ORD-003", "C102", "Keyboard",         79.99, "shipped",    "2025-02-01", "2025-03-12"),
    ("ORD-006", "C104", "Webcam ",            89.99, "cancelled",  "2025-03-01", "2025-03-12"),
    ("ORD-009", "C106", "SSD",              120.00, "processing", "2025-03-11", "2025-03-11"),
    ("ORD-010", "C107", " RAM 16GB",          65.00, "processing", "2025-03-12", "2025-03-12T08:00:00"),
    ("ORD-010", "C107", "RAM 16GB",          65.00, "shipped",    "2025-03-12", "2025-03-12T14:30:00"), # "duplicado", mas é um update do update :D
    ("ORD-011", "C108", "Docking Station",   None,  "pending",    "2025-03-12", "2025-03-12"),
    ("ORD-011", "C108", "Docking Station",   None,  None,    None, None)
]



In [96]:
orders1 = spark.createDataFrame(target_data, schema)
orders2 = spark.createDataFrame(incremental_data, schema)

In [97]:
# Data Standard enforcing

# Get all string columns
string_cols = [field.name for field in orders1.schema.fields if isinstance(field.dataType, StringType)]
string_cols2 = [field.name for field in orders2.schema.fields if isinstance(field.dataType, StringType)]

# Trim all string columns
orders1 = (orders1
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("cust_id", trim(col("cust_id")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("price", trim(col("price")))
    .withColumn("status", trim(col("status")))
    .withColumn("order_date", trim(col("order_date")))
    .withColumn("updated_at", trim(col("updated_at")))
)

orders2 = (orders2
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("cust_id", trim(col("cust_id")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("price", trim(col("price")))
    .withColumn("status", trim(col("status")))
    .withColumn("order_date", trim(col("order_date")))
    .withColumn("updated_at", trim(col("updated_at")))
)


# Cast order_date and updated_at
orders1 = (orders1
  .withColumn("order_date", to_date(col("order_date")))
  .withColumn("updated_at", to_timestamp(col("updated_at")))
)

orders2 = (orders2
  .withColumn("order_date", to_date(col("order_date")))
  .withColumn("updated_at", to_timestamp(col("updated_at")))
)

# Fill nulls with 'Unknown'
orders1 = orders1.na.fill("Unknown")
orders2 = orders2.na.fill("Unknown")

# Fill price nulls with 0
orders1 = orders1.na.fill(0, ["price"])
orders2 = orders2.na.fill(0, ["price"])

orders1.show()
orders2.show()

+--------+-------+------------+------+----------+----------+-------------------+
|order_id|cust_id|product_name| price|    status|order_date|         updated_at|
+--------+-------+------------+------+----------+----------+-------------------+
| ORD-001|   C100|      Laptop|1200.0|   shipped|2025-01-15|2025-01-16 00:00:00|
| ORD-002|   C101|       Mouse|  25.5| delivered|2025-01-20|2025-01-22 00:00:00|
| ORD-003|   C102|    Keyboard|  75.0|processing|2025-02-01|2025-02-01 00:00:00|
| ORD-004|   C100|     Monitor| 450.0|   shipped|2025-02-10|2025-02-11 00:00:00|
| ORD-005|   C103|  Headphones| 150.0| delivered|2025-02-14|2025-02-16 00:00:00|
| ORD-006|   C104|      Webcam| 89.99|processing|2025-03-01|2025-03-01 00:00:00|
| ORD-007|   C101|     USB Hub|  35.0|   shipped|2025-03-05|2025-03-06 00:00:00|
| ORD-008|   C105|      Tablet| 599.0|processing|2025-03-10|2025-03-10 00:00:00|
+--------+-------+------------+------+----------+----------+-------------------+

+--------+-------+---------

In [98]:
stacked = (
    orders1.withColumn("is_incremental", F.lit(0))
    .unionByName(
        orders2.withColumn("is_incremental", F.lit(1))
    )
)

w = Window.partitionBy("order_id").orderBy(F.col("is_incremental").desc(), F.col("updated_at").desc())

In [99]:
stacked.show()

+--------+-------+---------------+-------+----------+----------+-------------------+--------------+
|order_id|cust_id|   product_name|  price|    status|order_date|         updated_at|is_incremental|
+--------+-------+---------------+-------+----------+----------+-------------------+--------------+
| ORD-001|   C100|         Laptop| 1200.0|   shipped|2025-01-15|2025-01-16 00:00:00|             0|
| ORD-002|   C101|          Mouse|   25.5| delivered|2025-01-20|2025-01-22 00:00:00|             0|
| ORD-003|   C102|       Keyboard|   75.0|processing|2025-02-01|2025-02-01 00:00:00|             0|
| ORD-004|   C100|        Monitor|  450.0|   shipped|2025-02-10|2025-02-11 00:00:00|             0|
| ORD-005|   C103|     Headphones|  150.0| delivered|2025-02-14|2025-02-16 00:00:00|             0|
| ORD-006|   C104|         Webcam|  89.99|processing|2025-03-01|2025-03-01 00:00:00|             0|
| ORD-007|   C101|        USB Hub|   35.0|   shipped|2025-03-05|2025-03-06 00:00:00|             0|


In [100]:
merged = (
    stacked.withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn", "is_incremental")
)

merged.orderBy("order_id").show(truncate=False)

+--------+-------+---------------+-------+----------+----------+-------------------+
|order_id|cust_id|product_name   |price  |status    |order_date|updated_at         |
+--------+-------+---------------+-------+----------+----------+-------------------+
|ORD-001 |C100   |Laptop         |1200.0 |shipped   |2025-01-15|2025-01-16 00:00:00|
|ORD-002 |C101   |Mouse          |25.5   |delivered |2025-01-20|2025-01-22 00:00:00|
|ORD-003 |C102   |Keyboard       |79.99  |shipped   |2025-02-01|2025-03-12 00:00:00|
|ORD-004 |C100   |Monitor        |450.0  |shipped   |2025-02-10|2025-02-11 00:00:00|
|ORD-005 |C103   |Headphones     |150.0  |delivered |2025-02-14|2025-02-16 00:00:00|
|ORD-006 |C104   |Webcam         |89.99  |cancelled |2025-03-01|2025-03-12 00:00:00|
|ORD-007 |C101   |USB Hub        |35.0   |shipped   |2025-03-05|2025-03-06 00:00:00|
|ORD-008 |C105   |Tablet         |599.0  |processing|2025-03-10|2025-03-10 00:00:00|
|ORD-009 |C106   |SSD            |120.0  |processing|2025-03-11|2

In [101]:
stacked.printSchema()

root
 |-- order_id: string (nullable = false)
 |-- cust_id: string (nullable = false)
 |-- product_name: string (nullable = false)
 |-- price: string (nullable = false)
 |-- status: string (nullable = false)
 |-- order_date: date (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- is_incremental: integer (nullable = false)



In [108]:
merged.select("product_name", "status", "order_date", "updated_at").where(F.col("status") == "shipped").show()

+------------+-------+----------+-------------------+
|product_name| status|order_date|         updated_at|
+------------+-------+----------+-------------------+
|      Laptop|shipped|2025-01-15|2025-01-16 00:00:00|
|    Keyboard|shipped|2025-02-01|2025-03-12 00:00:00|
|     Monitor|shipped|2025-02-10|2025-02-11 00:00:00|
|     USB Hub|shipped|2025-03-05|2025-03-06 00:00:00|
|    RAM 16GB|shipped|2025-03-12|2025-03-12 14:30:00|
+------------+-------+----------+-------------------+

